# 01 · Baseline model — TF-IDF + linear classifier
**Owner:** P2 (Data & baseline) · **Goal:** first honest score, a saved model, and the artifacts P4's API needs.

Plan
1. Load data and build the model input (subject + text, cleaned)
2. Compare baselines for the **primary `category`**; check why train CV is over-optimistic; select on validation
3. Error analysis (confusion matrix, weakest classes)
4. Confidence check (the API scores calibration, so `confidence` must be honest)
5. Baseline heads for `secondary_category` and `is_urgent` (the response schema requires both)
6. Save model + `labels.json`, log the run to `experiments.md`

In [3]:
import sys, json, hashlib, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn

from sklearn.pipeline import Pipeline, make_union
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support)

# repo root = folder that contains app/ (works from training/ or the root)
ROOT = Path.cwd() if (Path.cwd() / "app").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from app.textproc import clean_text, build_input   # same cleaner the API will use

DATA_DIR  = ROOT / "data"          # git-ignored
MODEL_DIR = ROOT / "model"; MODEL_DIR.mkdir(exist_ok=True)
OUT_DIR   = ROOT / "training" / "outputs"; OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
warnings.filterwarnings("ignore")
print("scikit-learn", sklearn.__version__, "| root:", ROOT)

ImportError: cannot import name 'clean_text' from 'app.textproc' (c:\Users\ASUS\Documents\Personal Projects\vortex-ticket-classifier\app\textproc.py)

## 1. Load data
Same loading as the EDA (`keep_default_na=False` so empty subjects stay `""`).

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", keep_default_na=False)
val   = pd.read_csv(DATA_DIR / "validation.csv", keep_default_na=False)

for df in (train, val):
    df["model_text"] = [build_input(s, t) for s, t in zip(df.subject, df.text)]

CATEGORIES = sorted(train.category.unique())
print(train.shape, val.shape, "| classes:", len(CATEGORIES))
train[["model_text", "category"]].head(3)

## 2. Fixed category → team table (from the OpenAPI spec)

In [ ]:
CATEGORY_TO_TEAM = {
    "payment_refund":      "Payments & Refunds",
    "ride_trip_issue":     "Ride Operations",
    "lost_item":           "Lost & Found",
    "order_missing_wrong": "Food Operations",
    "delivery_delay":      "Delivery Operations",
    "food_quality":        "Restaurant Quality",
    "account_promo":       "Account Services",
    "safety_conduct":      "Trust & Safety",
    "app_technical":       "Tech Support",
    "general_inquiry":     "Front-line Support",
    "spam_irrelevant":     "Auto-close / Spam Filter",
}
assert set(CATEGORY_TO_TEAM) == set(CATEGORIES), "category set differs from the spec"

 ## 3. Features and candidate models
 
Tickets mix English, Sinhala, Tamil, Singlish and Tanglish with typos, so we use **word n-grams + character n-grams** (`char_wb` handles spelling variants and non-Latin scripts without a tokenizer).

In [ ]:
def make_features():
    return make_union(
        TfidfVectorizer(preprocessor=clean_text, analyzer="word", ngram_range=(1, 2),
                        min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b"),
        TfidfVectorizer(preprocessor=clean_text, analyzer="char_wb", ngram_range=(2, 5),
                        min_df=2, sublinear_tf=True, max_features=300_000),
    )

candidates = {
    "LogReg (C=10, balanced)": LogisticRegression(C=10, max_iter=2000, class_weight="balanced"),
    "LinearSVC (C=0.5, balanced)": LinearSVC(C=0.5, class_weight="balanced"),
}

X_tr, y_tr = train.model_text, train.category
X_va, y_va = val.model_text,   val.category
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

rows, fitted = [], {}
for name, clf in candidates.items():
    pipe = Pipeline([("features", make_features()), ("clf", clf)])
    t0 = time.time()
    cv = cross_val_score(pipe, X_tr, y_tr, cv=skf, scoring="f1_macro", n_jobs=-1)
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_va)
    rows.append({"model": name,
                 "cv_macro_f1_optimistic": cv.mean(), "cv_std": cv.std(),
                 "val_macro_f1": f1_score(y_va, pred, average="macro"),
                 "val_accuracy": accuracy_score(y_va, pred),
                 "secs": round(time.time() - t0, 1)})
    fitted[name] = pipe

results = pd.DataFrame(rows).round(4).set_index("model")
results

### Why CV and validation disagree (CV ≈ 0.99, validation ≈ 0.66)
Train tickets are built from a small pool of repeated sentences, so a random CV fold contains near-copies of its own training folds. Validation tickets are much less similar to train. Quick check: nearest-neighbour similarity inside train vs from validation to train.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
_v = TfidfVectorizer(preprocessor=clean_text, analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True).fit(pd.concat([X_tr, X_va]))
A, B = _v.transform(X_tr), _v.transform(X_va)
S_tt = cosine_similarity(A); np.fill_diagonal(S_tt, 0)
S_vt = cosine_similarity(B, A)
print(f"train -> train nearest neighbour: median sim {np.median(S_tt.max(1)):.2f}, share > 0.8: {(S_tt.max(1) > 0.8).mean():.0%}")
print(f"val   -> train nearest neighbour: median sim {np.median(S_vt.max(1)):.2f}, share > 0.8: {(S_vt.max(1) > 0.8).mean():.0%}")

**Consequence:** train-CV cannot be used to pick or tune models here. We select on **validation macro-F1** (the only honest estimate, and the closest match to the judges' hidden set) and report CV only as an optimistic reference. P3 should build a grouped CV or keep validation as the yardstick.

In [ ]:
best_name = results.val_macro_f1.idxmax()
best = fitted[best_name]
val_pred = best.predict(X_va)
print("Selected:", best_name)
print(classification_report(y_va, val_pred, digits=3))

## 4. Error analysis 

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_va, val_pred, labels=CATEGORIES, xticks_rotation=90,
                                        normalize="true", values_format=".2f", cmap="Blues", ax=ax, colorbar=False)
ax.set_title(f"Validation confusion matrix (row-normalised) — {best_name}")
plt.tight_layout(); plt.savefig(OUT_DIR / "baseline_confusion.png", dpi=150); plt.show()

cm = pd.DataFrame(confusion_matrix(y_va, val_pred, labels=CATEGORIES), index=CATEGORIES, columns=CATEGORIES)
pairs = (cm.where(~np.eye(len(cm), dtype=bool)).stack().sort_values(ascending=False).head(8))
print("Most confused (true -> predicted):"); print(pairs.to_string())

In [ ]:
# Per-language view: does the baseline struggle on any language/script?
perf = (val.assign(correct=(val_pred == y_va.values))
           .groupby("language").correct.agg(["mean", "size"]).round(3)
           .rename(columns={"mean": "accuracy", "size": "n"}))
perf

In [ ]:
# Look at a few mistakes (feeds P3's error analysis)
errs = val.assign(pred=val_pred)[val_pred != y_va.values]
errs[["language", "category", "pred", "text"]].head(10)

## 5. Confidence
The spec says `confidence` is used for **calibration**, so it must reflect real uncertainty. We use cross-validated probabilities via `CalibratedClassifierCV` around the winner and measure **ECE** (expected calibration error; lower is better).

def ece(y_true, proba, classes, bins=10):
    classes = np.array(classes)
    conf = proba.max(1); pred = classes[proba.argmax(1)]
    correct = (pred == np.asarray(y_true)).astype(float)
    edges = np.linspace(0, 1, bins + 1); total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return total

base_clf = candidates[best_name]
cal = Pipeline([("features", make_features()),
                ("clf", CalibratedClassifierCV(base_clf, method="sigmoid", cv=5))])
cal.fit(X_tr, y_tr)
proba = cal.predict_proba(X_va)
cal_pred = cal.classes_[proba.argmax(1)]

print(f"calibrated val macro-F1 : {f1_score(y_va, cal_pred, average='macro'):.4f}")
print(f"calibrated val accuracy : {accuracy_score(y_va, cal_pred):.4f}")
print(f"ECE (calibrated)        : {ece(y_va, proba, cal.classes_):.4f}")
conf = proba.max(1); ok = cal_pred == y_va.values
print(f"mean confidence when right: {conf[ok].mean():.3f} | when wrong: {conf[~ok].mean():.3f}")

In [ ]:
# Reliability diagram + how accuracy changes if we send low-confidence tickets to a human
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
edges = np.linspace(0, 1, 11); mids, accs = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (conf > lo) & (conf <= hi)
    if m.sum() >= 5: mids.append(conf[m].mean()); accs.append(ok[m].mean())
axes[0].plot([0, 1], [0, 1], "--", c="grey"); axes[0].plot(mids, accs, "o-")
axes[0].set(title="Reliability (validation)", xlabel="confidence", ylabel="accuracy")

ths = np.linspace(0.2, 0.9, 15)
axes[1].plot(ths, [ok[conf >= t].mean() if (conf >= t).any() else np.nan for t in ths], label="accuracy kept")
axes[1].plot(ths, [(conf >= t).mean() for t in ths], label="share auto-routed")
axes[1].set(title="Confidence threshold trade-off", xlabel="threshold"); axes[1].legend()
plt.tight_layout(); plt.savefig(OUT_DIR / "baseline_calibration.png", dpi=150); plt.show()

## 6. Heads for `secondary_category` and `is_urgent`
The response schema requires both keys, so the baseline needs a (simple) answer for each.

* **secondary_category** — only ~10% of tickets have one. We train a classifier over the 11 categories + `none`, then at prediction time enforce *secondary ≠ category* and *spam → null*.
* **is_urgent** — ~10% positive, and 75% of `safety_conduct` tickets are urgent. Binary classifier with balanced class weights; `spam_irrelevant` is forced to `False` (spec consistency rule).

In [ ]:
def simple_head(C=10):
    return Pipeline([("features", make_features()),
                     ("clf", LogisticRegression(C=C, max_iter=2000, class_weight="balanced"))])

# --- secondary category (label "none" when absent) ---
sec_tr = train.secondary_category.replace("", "none")
sec_va = val.secondary_category.replace("", "none")
sec_head = simple_head().fit(X_tr, sec_tr)
sec_raw = sec_head.predict(X_va)
print("secondary: exact-match accuracy incl. 'none' =", round(accuracy_score(sec_va, sec_raw), 4))
has2 = sec_va != "none"
print("secondary: recall on tickets that DO have one =", round((sec_raw[has2] == sec_va[has2]).mean(), 4))
print("secondary: false-positive rate on tickets with none =", round((sec_raw[~has2] != "none").mean(), 4))

# --- urgency ---
urg_head = simple_head().fit(X_tr, train.is_urgent)
urg_raw = urg_head.predict(X_va)
p, r, f, _ = precision_recall_fscore_support(val.is_urgent, urg_raw, average="binary")
print(f"is_urgent: precision={p:.3f} recall={r:.3f} F1={f:.3f}")

## 7. Prediction function with the consistency rules (prototype of `predictor.py`)

In [ ]:
def predict_tickets(cat_model, sec_model, urg_model, subjects, texts):
    inputs = [build_input(s, t) for s, t in zip(subjects, texts)]
    proba = cat_model.predict_proba(inputs)
    classes = cat_model.classes_
    cats = classes[proba.argmax(1)]; confs = proba.max(1)
    sec_proba = sec_model.predict_proba(inputs); sec_classes = sec_model.classes_
    urg = urg_model.predict(inputs)

    out = []
    for i, cat in enumerate(cats):
        # secondary: best non-"none" class that differs from category, only if "none" is not the top pick
        sec = None
        top = sec_classes[sec_proba[i].argmax()]
        if top != "none" and top != cat:
            sec = str(top)
        urgent = bool(urg[i])
        if cat == "spam_irrelevant":
            sec, urgent = None, False
        out.append({"category": str(cat), "secondary_category": sec,
                    "team": CATEGORY_TO_TEAM[cat], "is_urgent": urgent,
                    "confidence": round(float(confs[i]), 4)})
    return out

demo = predict_tickets(cal, sec_head, urg_head, val.subject.head(5), val.text.head(5))
pd.DataFrame(demo).assign(true=val.category.head(5).values)

In [ ]:
# Check consistency rules on the whole validation set
preds = pd.DataFrame(predict_tickets(cal, sec_head, urg_head, val.subject, val.text))
assert (preds.secondary_category.fillna("") != preds.category).all()
assert (preds.team == preds.category.map(CATEGORY_TO_TEAM)).all()
sp = preds.category == "spam_irrelevant"
assert (~preds[sp].is_urgent).all() and preds[sp].secondary_category.isna().all()
assert preds.confidence.between(0, 1).all()
print("consistency rules OK on", len(preds), "validation tickets")

## 8. Final fit and save
Scores above come from **train → validation**. For the shipped model we **refit on train + validation** (more data, same settings). Set `REFIT_ON_FULL = False` to ship the exact model that was scored.

In [ ]:
REFIT_ON_FULL = True

if REFIT_ON_FULL:
    full = pd.concat([train, val], ignore_index=True)
    Xf = full.model_text
    cat_final = Pipeline([("features", make_features()),
                          ("clf", CalibratedClassifierCV(base_clf, method="sigmoid", cv=5))]).fit(Xf, full.category)
    sec_final = simple_head().fit(Xf, full.secondary_category.replace("", "none"))
    urg_final = simple_head().fit(Xf, full.is_urgent)
else:
    cat_final, sec_final, urg_final = cal, sec_head, urg_head

bundle = {"category": cat_final, "secondary": sec_final, "urgent": urg_final}
model_path = MODEL_DIR / "model.joblib"
joblib.dump(bundle, model_path, compress=3)

sha = hashlib.sha256(model_path.read_bytes()).hexdigest()[:8]
MODEL_VERSION = f"baseline-tfidf-{sha}"

labels = {
    "model_version": MODEL_VERSION,
    "sklearn_version": sklearn.__version__,
    "categories": [str(c) for c in cat_final.classes_],
    "category_to_team": CATEGORY_TO_TEAM,
    "refit_on_train_plus_val": REFIT_ON_FULL,
}
(MODEL_DIR / "labels.json").write_text(json.dumps(labels, indent=2, ensure_ascii=False), encoding="utf-8")
print(MODEL_VERSION, "|", round(model_path.stat().st_size / 1e6, 1), "MB")

In [ ]:
# Reload from disk exactly like the API will, and check it reproduces predictions
loaded = joblib.load(model_path)
check = predict_tickets(loaded["category"], loaded["secondary"], loaded["urgent"], val.subject.head(50), val.text.head(50))
ref   = predict_tickets(cat_final, sec_final, urg_final, val.subject.head(50), val.text.head(50))
assert check == ref
print("reload OK")

## 9. Log the run (feeds `training/experiments.md` and the report)

In [ ]:
sel = results.loc[best_name]
sec_recall = float((sec_raw[has2] == sec_va[has2]).mean())
line = (f"| baseline-01 | {MODEL_VERSION} | word(1,2)+char_wb(2,5) TF-IDF + {best_name}, sigmoid-calibrated | "
        f"{sel.cv_macro_f1_optimistic:.4f} | {sel.val_macro_f1:.4f} | {sel.val_accuracy:.4f} | "
        f"urgent F1 {f:.3f}; secondary recall {sec_recall:.3f}; ECE {ece(y_va, proba, cal.classes_):.3f} |")

exp = ROOT / "training" / "experiments.md"
if not exp.exists():
    exp.write_text("# Experiment log\n\n| id | model_version | setup | train-CV macro-F1 (optimistic) | val macro-F1 | val acc | notes |\n"
                   "|---|---|---|---|---|---|---|\n", encoding="utf-8")
with exp.open("a", encoding="utf-8") as fh:
    fh.write(line + "\n")
print(line)